# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mishellscripts/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content item on one date for a particular client. The table covers dates ranging from 2025-01-27 to 2026-06-30.

In [5]:
con.sql(f"""
    SELECT MIN(report_date) AS start, MAX(report_date) AS end,
           DATEDIFF('month', MIN(report_date), MAX(report_date)) AS months_of_history
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬───────────────────┐
│   start    │    end     │ months_of_history │
│    date    │    date    │       int64       │
├────────────┼────────────┼───────────────────┤
│ 2025-01-27 │ 2026-06-30 │                17 │
└────────────┴────────────┴───────────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Feature - gsc_impressions, gsc_clicks, gsc_avg_position, gsc_clicks from fact table. content_updated_date, content_type from content dimension table.
- Label - is_declining, is_recovered
- Context - client_hash_id and content_hash_id for joining
- Excluded - ga4 data available due to low availability,trend_pct due to leakage, and data from last 30 days in favor of fine-grained data.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
con.sql(f"""
    SELECT SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END)*100.0/COUNT(*) AS ga4_data_available_pct
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────┐
│ ga4_data_available_pct │
│         double         │
├────────────────────────┤
│      3.572564977103317 │
└────────────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The data can never tell the actual cause of recovery or decline. It cannot predict anything using ga4 features. Clients that do not have enough history for training will be excluded from analysis, which can lead to bias if model is used on similar clients.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.